### Exercise 2: Convolutional Neural Networks

In this exercise, you will train a convolutional neural network (CNN) for **image denoising**—removing noise from images to recover clean images. Noise can arise from poor lighting, camera sensors, or compression artifacts, and the goal is to reduce this noise while preserving image details.

<div style="text-align: center;">
    <img src="denoising_example.PNG" alt="Image Title" style="width:500px; height:auto;"/>
</div>

For supervised learning, we will use pairs of noisy images and their original clean versions as inputs and labels respectively. To simulate this, we will add noise to clean images, treating the noisy images as inputs and the clean images as targets.

You will build a CNN with a U-Net architecture to perform image denoising on the [STL10 dataset](https://cs.stanford.edu/~acoates/stl10/).

#### Code Template

To complete this assignment, you will fill in the provided code template below. Your main tasks include:

1. **Defining the Model (10 points)**
2. **Setting the Loss Criterion and Optimizer (2 points)**
3. **Writing the Training Loop (4 points)**
4. **Writing the Testing Loop and checking your trained model (4 points)**

Make sure to follow the instructions in the code template carefully and test your implementation before submission. If your code passes all the visible test cases, you get 50% of the points for each task. The remaining points are based on hidden test cases evaluated after submission.

**Deliverables:** <br>

Submit the completed notebook (ex2.ipynb) and your trained model (best_model.pth) to moodle. The autograder does not train your model: it grades the best_model.pth you submit, so submit the model trained with your final code.
Do not change the name of the notebook file. It may result in 0 points for the exercise.

**Note:** The variable `feedback_txt` is preserved for autograding purposes only. Do not modify or delete it. `skip_training` is also used for autograding, you may modify its value in the cell where it is defined, but do not add or modify it anywhere else in the notebook. You will lose points if there are problems related to these variables.

**Let's begin!**

In [ ]:
skip_training = False 

In [ ]:
# Do not delete this cell


In [ ]:
# Do not delete this cell


<h3>Import all the necessary libraries </h3>

First, we will import the essential libraries needed for building and training a deep learning model using PyTorch:

- `torch`: The core library for building neural networks in PyTorch.
- `torch.nn`: Provides modules and classes for building neural network layers.
- `torch.optim`: Provides classes for various optimizers used to update weights during training.
- `numpy`: A library for numerical computations, particularly for working with arrays.
- `matplotlib.pyplot`: Used for plotting graphs and visualizing data.
- `torchvision`: A PyTorch library for image transformations and loading datasets.
- `transforms`: A module within `torchvision` for applying transformations to images, such as resizing, normalization, etc.
- `STL10`: A popular image dataset used in deep learning tasks.
- `random_split`: A utility to split datasets into training, validation, and test sets.
- `DataLoader`: Helps in efficiently loading datasets in batches for model training.


In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import torchvision
from torchvision import transforms
from torchvision.datasets import STL10
from torch.utils.data import random_split, DataLoader

import warnings
warnings.filterwarnings("ignore")

feedback_txt = [] # preserved for autograding purposes only. Do not modify or delete it.

In [ ]:
# Helpers for grading and for the test cells. Do not modify or delete this cell.
import shutil
import tempfile
from contextlib import ExitStack, contextmanager
from unittest.mock import patch
from torch.utils.data import TensorDataset

# True while the autograder runs this notebook
AUTOGRADING = os.environ.get('NBGRADER_EXECUTION') == 'autograde' or globals().get('autograding', False)

# The graded weights. During grading they are copied before any of your code runs, so nothing can overwrite them.
SUBMITTED_WEIGHTS = 'best_model.pth'
if AUTOGRADING and os.path.exists(SUBMITTED_WEIGHTS):
    SUBMITTED_WEIGHTS = shutil.copy(SUBMITTED_WEIGHTS, tempfile.mkdtemp())

tests_passed, tests_failed = [], []    # for the feedback


def record_test_pass(name):
    tests_passed.append(name)


def record_test_failure(name, error):
    """Add a line about a failed test to feedback_txt."""
    tests_failed.append(name)
    if isinstance(error, AssertionError):
        message = str(error)
    elif isinstance(error, Exception):
        message = f"your code raised {type(error).__name__}" + (f": {error}" if str(error) else ".")
    else:    # e.g. the cell was stopped because it ran too long
        message = f"the test was stopped ({type(error).__name__}), probably because it ran too long."
    feedback_txt.append(f"{name}: {message}")


def load_weights(path):
    """Load a saved state_dict, also one saved from a model wrapped by torch.compile or nn.DataParallel."""
    state_dict = torch.load(path, map_location=device)
    if isinstance(state_dict, nn.Module):    # the whole model was saved
        state_dict = state_dict.state_dict()
    for prefix in ('_orig_mod.', 'module.'):
        state_dict = {(key[len(prefix):] if key.startswith(prefix) else key): value for key, value in state_dict.items()}
    return state_dict


@contextmanager
def in_scratch_folder():
    """Run code in a temporary folder, so that files it writes (e.g. best_model.pth) are thrown away."""
    folder, scratch = os.getcwd(), tempfile.mkdtemp()
    os.chdir(scratch)
    try:
        yield
    finally:
        os.chdir(folder)
        shutil.rmtree(scratch, ignore_errors=True)


class StandIn(nn.Module):
    """A tiny fixed model, used to test train() and test() independently of your U-Net."""
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 3, kernel_size=1)
        with torch.no_grad():
            self.conv.weight.copy_(4 * torch.eye(3).view(3, 3, 1, 1))
            self.conv.bias.fill_(-2.)
        self.inputs = []          # the input of each forward pass
        self.grad_enabled = []    # whether gradient tracking was on in each forward pass

    def forward(self, x):
        self.inputs.append(x.detach().clone())
        self.grad_enabled.append(torch.is_grad_enabled())
        return torch.sigmoid(self.conv(x))


def stand_in_model_and_data():
    """The stand-in model, and 16 fixed random images in 2 batches with clearly different losses."""
    images = torch.rand(16, 3, 96, 96, generator=torch.Generator().manual_seed(0))
    images[8:] *= 0.3
    return StandIn().to(device), DataLoader(TensorDataset(images, torch.zeros(16)), batch_size=8)


def fixed_noise(images, mean=0, std=0.1):
    """A deterministic stand-in for add_gaussian_noise, used by the tests."""
    return images * 0.5


<h3> Select the device to run the model.</h3>

In this step, we will determine whether the model will run on a GPU (if available) or fall back to the CPU. Using a GPU for deep learning tasks can significantly speed up training, especially with large models or datasets.

  - If a MPS-enabled device is available (Mac OS), it assigns `'mps'` as the device.
  - If a CUDA-enabled GPU is available, it assigns `'cuda'` as the device.
  - If no GPU is available, it defaults to the CPU (`'cpu'`).

Make sure to always transfer both the model and the data to the selected device. (We will do this in the later steps.)


In [ ]:
if torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

<h3> Download the dataset and create a dataloader to loop through mini batches of data. </h3>

In this step, we will download and prepare the STL10 dataset, which consists of 96x96 pixel images divided into 10 classes. We will split the dataset into a training set and a test set.

- `STL10`: Downloads the dataset and applies the transformation to convert the images into tensors.
- `train_size` and `test_size`: Specify the number of samples for the training and testing datasets, respectively. Here, we're using 4000 samples for training and 1000 for testing.
- `random_split`: Randomly splits the dataset into the specified sizes for training and testing. The split uses a fixed seed, so the test images are the same on every run and at grading.
- `DataLoader`: Loads the dataset into batches, making it easier to iterate through the dataset during training and testing. 
  - `batch_size=8`: Each batch will contain 8 images.
  - `shuffle=True`: Randomly shuffles the dataset at each epoch to improve training.

Be patient while the dataset downloads for the first time. torchvision downloads the whole STL10 archive (about 2.6 GB, which can take an hour on a slow connection) and unpacks it, so you need about 5.7 GB of free disk space. We only use the 5000 labelled training images. After the first successful run you can delete `stl10_binary.tar.gz` in the dataset folder to free 2.6 GB.

In [ ]:
dataset_path = './dataset_ex2' # you can change the path if you want to store the dataset somewhere else.

In [ ]:
# A TA sets the dataset folder for grading; otherwise dataset_path from the cell above is used
dataset_path = os.environ.get('EX2_DATASET_PATH') or globals().get('TA_dataset_path') or globals().get('dataset_path', './dataset_ex2')
try:
    # The autograder never downloads the dataset
    dataset = STL10(root=dataset_path, split='train', transform=transforms.ToTensor(), download=not AUTOGRADING)
except BaseException as e:
    feedback_txt.append(f"Loading the STL10 dataset failed ({type(e).__name__}: {e}). Please contact your TA.")
    raise
train_size = 4000
test_size = 1000

# A fixed seed, so that the test images are the same on every run and at grading
train_dataset, test_dataset = random_split(dataset, [train_size, test_size], generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=True)

<h3> Adding Gaussian Noise to Images </h3>

In this step, we will create a function to add Gaussian noise to the images and visualize both the original and noisy images.
- `add_gaussian_noise(images, mean, std)`: Adds random Gaussian noise to the images.
  - `mean` and `std`: These parameters define the mean and standard deviation of the noise.
  - `torch.randn_like(images)`: Generates random noise of the same shape as the input images.
  - `torch.clamp`: Ensures the noisy pixel values stay between 0 and 1, as required for valid image data.
<br><br>
- `imshow(img, ax, title)`: A utility function to display images.
  - The images are converted from tensors to NumPy arrays, and their channels are rearranged for correct visualization.
<br><br>
- We then retrieve a batch of training data from `train_loader` and apply Gaussian noise to this batch.
- The original and noisy images are displayed side by side using `matplotlib` for comparison.

In [ ]:
def add_gaussian_noise(images, mean=0, std=0.1):
    noise = torch.randn_like(images) * std + mean
    noisy_images = images + noise
    noisy_images = torch.clamp(noisy_images, 0., 1.)
    return noisy_images

In [ ]:
def imshow(img, ax, title):
    np_img = img.numpy()
    ax.imshow(np.transpose(np_img, (1, 2, 0)))
    ax.set_title(title)
    ax.axis('off')

The STL10 dataset is mainly built for classification task. For our purpose, we shall ignore the classification labels and use only the images. Hence we use:
- `images, _ = next(iter(train_loader))`: We only store the images (images) and ignore ( _ ) the labels.

In [ ]:
# Get a batch of training data
images, _ = next(iter(train_loader))
noisy_images = add_gaussian_noise(images)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
imshow(torchvision.utils.make_grid(images[:4]), axes[0], 'Original Images')
imshow(torchvision.utils.make_grid(noisy_images[:4]), axes[1], 'Noisy Images')
plt.show()

<h3>Task 1: Define the U-Net Model Class <b> (10 points) </b>. </h3>

Your task is implementing a mini version of the U-Net architecture shown below.

<div style="text-align: center;">
    <img src="MiniUNet.png" alt="Image Title" style="width:500px; height:auto;"/>
</div>

**Your Task:**
- Initialize all the layers in the `__init__` method.
- Complete the `forward` method to pass the input through the layers in accordance with the architecture.

**Useful links:**

- [Conv2D in PyTorch](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html)
- [ReLU in PyTorch](https://pytorch.org/docs/stable/generated/torch.nn.ReLU.html)
- [MaxPool2D in PyTorch](https://pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html)
- [Transpose Conv2D in PyTorch](https://pytorch.org/docs/stable/generated/torch.nn.ConvTranspose2d.html)
- [Sigmoid in PyTorch](https://pytorch.org/docs/stable/generated/torch.nn.Sigmoid.html)


In [ ]:
class MiniUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = nn.Sequential(
            nn.Conv2d(in_channels = 3, out_channels = 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2) # can be reused in the forward as it has no parameters.

        self.enc2=None
        self.enc3= None
        # YOUR CODE HERE
        raise NotImplementedError()

        
        # Decoding path
        self.upconv1 = nn.ConvTranspose2d(in_channels = 256, out_channels = 128, kernel_size=2, stride=2)
        self.dec1 = nn.Sequential(
            nn.Conv2d(in_channels = 256, out_channels = 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )

        self.upconv2=None
        self.dec2= None
        self.dec3=None
        
        # YOUR CODE HERE
        raise NotImplementedError()

    def forward(self, x):
        """ 
        This function implements the forward pass
        
        Parameters:
            x: input images of shape (batch_size, 3, 96, 96)
        Returns: 
            denoised images of shape (batch_size, 3, 96, 96)
        """
        # YOUR CODE HERE
        raise NotImplementedError()
        

**Test the correctness of your model by running the cell below**

In [ ]:
# test the model definition
def check_model_visible():
    model = MiniUNet()
    conv2d_count = sum(isinstance(layer, nn.Conv2d) for layer in model.modules())
    upconv2d_count = sum(isinstance(layer, nn.ConvTranspose2d) for layer in model.modules())
    problems = []
    if conv2d_count != 6:
        problems.append(f"Expected 6 Conv2d layers, got {conv2d_count}.")
    if upconv2d_count != 2:
        problems.append(f"Expected 2 ConvTranspose2d layers, got {upconv2d_count}.")
    try:
        output = model(torch.randn(1, 3, 96, 96))
        large_input_output = model(100 * torch.randn(1, 3, 96, 96))
    except Exception as e:
        problems.append(f"Passing an input of shape (1, 3, 96, 96) through your model raised {type(e).__name__}: {e}")
        output = None
    if output is not None and output.shape != (1, 3, 96, 96):
        problems.append(f"Expected output shape (1, 3, 96, 96), instead got {tuple(output.shape)}.")
    elif output is not None and not all(torch.all((o >= 0) & (o <= 1)) for o in (output, large_input_output)):
        problems.append("The output is not within the expected range [0, 1]. Sigmoid might be missing.")
    assert not problems, " ".join(problems)

try:
    check_model_visible()
except BaseException as error:
    record_test_failure("Model definition: visible test", error)
    raise
record_test_pass("Model definition: visible test")
print(f"\033[92m{'Good job! All visible tests passed! You can proceed further.'}\033[0m")


In [ ]:
# This cell contains hidden test cases that will be evaluated after submission. DO NOT DELETE THE CELL.


<h3> Task 2: Define the loss function and optimizer. <b> (2 Points) </b></h3>

You will implement the `loss_and_optimizer` function, which is essential for training your convolutional neural network (CNN). Follow these guidelines to complete this function:

1. **Choose the Loss Function**: 
   - Since this exercise involves image denoising, the goal is to minimize the difference between the denoised output and the original clean image. 
   - A common choice for this task is the Mean Squared Error (MSE) loss, which measures the average squared difference between the predicted and target values.
   - [Refer: MSE loss in PyTorch](https://pytorch.org/docs/stable/generated/torch.nn.MSELoss.html#torch.nn.MSELoss)

2. **Select the Optimizer**: 
   - An optimizer is used to update the model parameters based on the loss function. 
   - The Adam optimizer is widely used for its efficiency and effectiveness in handling large datasets and complex models. Initialize the Adam optimizer and set the learning rate to 0.001 to control the step size during optimization.
   - [Refer: Adam optimizer in PyTorch](https://pytorch.org/docs/stable/generated/torch.optim.Adam.html#torch.optim.Adam)

3. **Return Both Components**: 
   - Your function should return both the loss criterion and the optimizer.


In [ ]:
def loss_and_optimizer(model):
    """
    This function initializes and returns the loss criterion and optimizer for training a CNN model.

    Parameters:
    model: The model to be trained.

    Returns:
    tuple: A tuple containing:
        -  The loss function (Mean Squared Error loss).
        -  The optimizer (Adam) for updating model parameters.
    """
    
    # YOUR CODE HERE
    raise NotImplementedError()
    

In [ ]:
model = MiniUNet()
criterion, optimizer = loss_and_optimizer(model)

model = model.to(device) 
criterion = criterion.to(device)

**Test the correctness of your loss_and_optimizer function by running the cell below**

In [ ]:
# This cell tests the correctness of loss function and optimizer (with a small stand-in model)
def check_loss_and_optimizer_visible():
    stand_in, _ = stand_in_model_and_data()
    criterion, optimizer = loss_and_optimizer(stand_in)
    assert not (isinstance(criterion, torch.optim.Optimizer) and isinstance(optimizer, nn.Module)), \
        "loss_and_optimizer() should return the criterion first and then the optimizer."
    assert isinstance(criterion, nn.MSELoss), "The criterion is not set properly to MSE loss."
    optimizer_params = {id(p) for group in optimizer.param_groups for p in group['params']}
    assert optimizer_params == {id(p) for p in stand_in.parameters()}, "The optimizer does not contain the model parameters."

try:
    check_loss_and_optimizer_visible()
except BaseException as error:
    record_test_failure("Loss and optimizer: visible test", error)
    raise
record_test_pass("Loss and optimizer: visible test")
print(f"\033[92m{'Good job! All visible tests passed!'}\033[0m")


In [ ]:
# This cell contains hidden test cases that will be evaluated after submission


<h3> Task 3: Training loop. <b>(4 points)</b></h3>

In this step, we will implement the training loop for our U-Net model. The training loop is where the model learns from the training data by adjusting its weights based on the loss function.

#### Key Components of the Training Loop:

- `num_epochs = 10`: This specifies the number of times the entire training dataset will pass through the model. Training 10 epochs takes about half an hour on a laptop CPU, and much less on a GPU. The training cell below saves the trained model to `best_model.pth`. At grading, that cell does not train: it loads your `best_model.pth`.
- `model.train()`: Sets the model to training mode, which activates certain layers (like dropout) that are only active during training.
- `for epoch in range(num_epochs)`: Iterates over the specified number of epochs.
   
- `for i, (images, _) in enumerate(train_loader)`: Loops through batches of training data.
- `noisy_images = add_gaussian_noise(images)`: Applies Gaussian noise to the images to simulate noisy inputs.

**Your task is to complete the train function using the guidelines below:**

- set the gradients to zero.
- pass the input to the model to get the predictions.
- calculate the MSE between the predicted image and the label (the clean image).
- calculate the gradients with respect to the loss.
- update the model weights using the optimizer.
- append the loss value to the `epoch_loss` list for every iteration.




In [ ]:
# Training loop

def train(model, optimizer, criterion, train_loader, num_epochs=10, verbose=True): 
    """ 
    Function to train the model.
    
    Parameters: 
    - model: The model to train.
    - optimizer: The optimizer for updating model parameters.
    - criterion: The loss function to use.
    - train_loader: DataLoader for training data.
    - num_epochs: Number of training epochs (default is 10).
    - verbose: Boolean: print training progress and loss (default is True) 
    """
    model.train()
    for epoch in range(num_epochs):
        epoch_loss = []                                  # Append the loss of every iteration to the epoch loss
        for i, (images, _) in enumerate(train_loader):
            
            images = images.to(device)                  # Move the input to 'device' (CPU or GPU)
            noisy_images = add_gaussian_noise(images)   # Add Gaussian noise to simulate noisy images
    
            # YOUR CODE HERE
            raise NotImplementedError()

        if verbose:
            print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {np.mean(epoch_loss):.4f}') 

**Test the correctness of your train function by running the cell below**

In [ ]:
# This cell tests the training code (with a small stand-in model instead of your U-Net)
def check_train_visible():
    stand_in, loader = stand_in_model_and_data()
    optimizer = optim.Adam(stand_in.parameters(), lr=0.001)
    weights_before = [p.detach().clone() for p in stand_in.parameters()]
    backward_passes = []
    handles = [p.register_hook(lambda grad: backward_passes.append(1)) for p in stand_in.parameters()]
    with in_scratch_folder():
        train(stand_in, optimizer, nn.MSELoss(), loader, num_epochs=1, verbose=False)
    for handle in handles:
        handle.remove()
    if all(torch.equal(before, p.detach()) for before, p in zip(weights_before, stand_in.parameters())):
        if not backward_passes:
            raise AssertionError("The weights did not change and no gradients were computed. You forgot to calculate the gradients.")
        raise AssertionError("The weights did not change, although gradients were computed. You forgot to update the weights, "
                             "or you set the gradients to 0 between calculating them and updating the weights.")

try:
    check_train_visible()
except BaseException as error:
    record_test_failure("Training loop: visible test", error)
    raise
record_test_pass("Training loop: visible test")
print(f"\033[92m{'Good job! you can now proceed to train your model.'}\033[0m")


In [ ]:
# This cell contains hidden test cases that will be evaluated after submission


In [ ]:
num_epochs = 10
if AUTOGRADING or os.environ.get('NBGRADER_EXECUTION') == 'validate':
    skip_training = True    # the autograder never trains: it grades the best_model.pth you submit
if not skip_training:
    train(model, optimizer, criterion, train_loader, num_epochs)
    torch.save(model.state_dict(), 'best_model.pth')
    print("Your trained model is saved successfully!")
else:
    model.load_state_dict(load_weights(SUBMITTED_WEIGHTS))
    print("Loaded weights from your saved model successfully!")

<h3>Task 4: Testing Loop and your trained model. <b> (4 points) </b></h3>
In this step, we will evaluate the performance of the trained U-Net model on the test dataset. The testing loop is used to assess how well the model performs on unseen data.

#### Key Components of the Testing Loop:

- `model.eval()`: Sets the model to evaluation mode. This is important because it deactivates certain layers that behave differently during training (e.g., batch normalization layers, dropout layers).

**Your task**:
   - Complete the `test` function below using the `train` function as a reference. Think about which steps from the `train` function are necessary and which are not.
   - Check PyTorch documentation to figure out if there are any ways to reduce memory consumption during inference.

The tests of your `test` function (3 points) use a small stand-in model, so they do not depend on your U-Net. The last test cell (1 point) checks how well your saved model denoises.

In [ ]:
# Testing loop
def test(model, criterion, test_loader):
    """ 
    Function to test the model.
    
    Parameters: 
    - model: The model to test.
    - criterion: The loss function to calculate test loss.
    - test_loader: DataLoader for testing data.
    """
    model.eval()
    test_loss = []
    # YOUR CODE HERE
    raise NotImplementedError()
    return np.mean(test_loss)

In [ ]:
test_loss = test(model, criterion, test_loader)
print(f'Test loss: {test_loss:.4f}')

**Test the correctness of your test function by running the cell below**

In [ ]:
# This cell contains visible test cases for the 'test' function (with a small stand-in model instead of your U-Net)
def check_test_visible():
    stand_in, loader = stand_in_model_and_data()
    clean_batches = [images.to(device) for images, _ in loader]
    with ExitStack() as stack:
        stack.enter_context(in_scratch_folder())
        mock_add_noise = stack.enter_context(patch('__main__.add_gaussian_noise', side_effect=fixed_noise))
        trained_optimizer = globals().get('optimizer')
        optimizer_mocks = [stack.enter_context(patch.object(trained_optimizer, name)) for name in ('zero_grad', 'step')] \
            if isinstance(trained_optimizer, torch.optim.Optimizer) else []
        result = test(stand_in, nn.MSELoss(), loader)
    model_inputs = stand_in.inputs
    assert len(model_inputs) == len(clean_batches), \
        f"test() should pass each of the {len(clean_batches)} test batches through the model once, but it called the model {len(model_inputs)} times."
    assert not any(torch.allclose(x, c) for x, c in zip(model_inputs, clean_batches)), \
        "Did you add noise to the images before passing it to the model?"
    if mock_add_noise.called:
        assert all(torch.allclose(x, fixed_noise(c)) for x, c in zip(model_inputs, clean_batches)), \
            "Add noise to the images once, with add_gaussian_noise, before passing them to the model."
    assert not any(m.called for m in optimizer_mocks), "Think about the purpose of optimizers during inference."
    with torch.no_grad():
        expected = np.mean([nn.functional.mse_loss(StandIn().to(device)(x), c).item() for x, c in zip(model_inputs, clean_batches)])
    try:
        result = float(result)
    except (TypeError, ValueError):
        raise AssertionError(f"test() should return the average test loss as a number, but it returned {result!r}.")
    assert abs(result - expected) <= 1e-6 + 1e-3 * expected, \
        f"test() should return the average loss over the test batches: expected {expected:.6f}, got {result:.6f}."

try:
    check_test_visible()
except BaseException as error:
    record_test_failure("Testing loop: visible test", error)
    raise
record_test_pass("Testing loop: visible test")
print(f"\033[92m{'Great job! You passed all the visible test cases!'}\033[0m")


In [ ]:
# This cell contains hidden test cases that will be evaluated after submission


**Check how well your saved model denoises (1 point)**

The cell below loads your saved model from `best_model.pth`, the file you submit, and computes its test loss (MSE) on the first 200 test images, with fixed noise. It uses the same images and noise every time, so you get the same result here as at grading. The test loss must be below 0.002.

If it is not, check Tasks 1-3 and train for more epochs: run the training cell again (it continues training your model and saves it to `best_model.pth`), then run this cell again.

In [ ]:
# This cell checks how well your saved model (best_model.pth) denoises
def official_test_loss(model, n_images=200, batch_size=50):
    """MSE of the denoised images on the first n_images test images, with fixed noise."""
    noise_generator = torch.Generator().manual_seed(2026)
    model.eval()
    losses = []
    with torch.no_grad():
        for start in range(0, n_images, batch_size):
            images = torch.stack([test_dataset[i][0] for i in range(start, start + batch_size)])
            noisy_images = torch.clamp(images + 0.1 * torch.randn(images.shape, generator=noise_generator), 0., 1.)
            outputs = model(noisy_images.to(device))
            losses.append(nn.functional.mse_loss(outputs, images.to(device)).item())
    return float(np.mean(losses))


def check_trained_model():
    assert 'test_dataset' in globals(), "The test images are not available because loading the dataset failed (see above)."
    saved_model = MiniUNet().to(device)
    try:
        state_dict = load_weights(SUBMITTED_WEIGHTS)
    except FileNotFoundError:
        raise AssertionError("best_model.pth was not found. Train your model with the training cell, which saves it, "
                             "and submit best_model.pth with the notebook.")
    except Exception as e:
        raise AssertionError(f"best_model.pth could not be read ({type(e).__name__}). Save your model with the training cell, "
                             "which uses torch.save(model.state_dict(), 'best_model.pth').")
    try:
        saved_model.load_state_dict(state_dict)
    except Exception as e:
        raise AssertionError(f"best_model.pth does not fit your MiniUNet ({(str(e).strip().splitlines() or [''])[0][:200]}). "
                             "Did you change the model after training? Train it again with the training cell.")
    loss = official_test_loss(saved_model)
    print(f"Test loss of your saved model (best_model.pth): {loss:.5f} (it must be below 0.002)")
    assert loss < 0.002, (f"The test loss of your saved model (best_model.pth) is {loss:.5f}, but it should be below 0.002. "
                          "Check Tasks 1-3 and train for more epochs.")

try:
    check_trained_model()
except BaseException as error:
    record_test_failure("Trained model", error)
    raise
record_test_pass("Trained model")
print(f"\033[92m{'Great job! Your saved model denoises well.'}\033[0m")


In [ ]:
def visualize_results(model, test_loader):
    """
    Function to visualize the denoiser
    Parameters:
    - model: The model to use for denoising
    - test_loader: The DataLoader for testing data
    """

    # get one batch of images
    images, _ = next(iter(test_loader))
    images = images.to(device)

    # Add Gaussian noise
    noisy_images = add_gaussian_noise(images)

    #Forward pass to get the denoised images
    with torch.no_grad():
        outputs = model(noisy_images)

    # Display original, noisy, and denoised images
    fig, axs = plt.subplots(1, 3, figsize=(15, 5))               
    axs[0].imshow(np.transpose(images[0].cpu().numpy(), (1, 2, 0)))
    axs[0].set_title('Original')
    axs[1].imshow(np.transpose(noisy_images[0].cpu().numpy(), (1, 2, 0)))
    axs[1].set_title('Noisy')
    axs[2].imshow(np.transpose(outputs[0].cpu().numpy(), (1, 2, 0)))
    axs[2].set_title('Denoised')
    plt.show()
    return

In [ ]:
# visualize the results. You can run this cell multiple times to see different test images.
visualize_results(model, test_loader)

In [ ]:
# Do not delete this cell
